# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/h0pi/flyrank-ml-intenrship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**The contract in five plain answers**

1. **What one row means.** One content page (client × content item) that had Google Search data in March 2026, seen at the decision moment: the morning of **2026-03-16**. The raw table is one row per page per day; I add those days up into one row per page.
2. **Which tables.** `fact_content_daily_performance`, only the `month=2026-03` partition (daily search numbers), and `dim_content` (the page's creation date).
3. **Which time window.** Features come from **2026-03-01 to 2026-03-15**. The label comes from **2026-03-16 to 2026-03-31**. The final month (June 2026) stays untouched as a sealed test month.
4. **What I predict.** `is_declining` = 1 when the page's average daily impressions in the second half of March fell below 80% of its average daily impressions in the first half. Pages are ranked by the predicted chance of decline, so a content team can review the top of the list first (Precision@50, as in week 2).
5. **What I deliberately exclude.** `content_updated_date` and `last_optimized_date`. They are stored as of the export (July 2026), so they can contain edits made *after* 2026-03-16, which is information nobody had at the decision moment.

The cell below connects to the release and keeps a local copy of the few March columns I need, so every query after it runs on the same data without hitting Hugging Face rate limits.

In [2]:
%pip -q install duckdb huggingface_hub

import os, getpass
import duckdb
import numpy as np
import pandas as pd

# Token order: env var -> Colab Secret -> prompt. Never paste a token into a cell (public repo).
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# One download of only the columns this notebook touches.
con.execute(f'''
    CREATE OR REPLACE TABLE march AS
    SELECT report_date, client_hash_id, content_hash_id,
           gsc_data_available, ga4_data_available,
           gsc_impressions, gsc_clicks, gsc_avg_position
    FROM {MONTH}
''')
con.execute(f'''
    CREATE OR REPLACE TABLE content AS
    SELECT client_hash_id, content_hash_id, content_created_date
    FROM {DIM_CONTENT}
''')
print('loaded March 2026 into the local table "march"')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

loaded March 2026 into the local table "march"


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (1–15 March) | **Feature inputs** | Observed before the decision moment |
| `content_created_date` | **Feature input** | Fixed when the page is created, long before 16 March |
| `gsc_impressions` (16–31 March) | **Label** | The future window I predict. Never a feature |
| `report_date` | **Context** | Decides which window a day belongs to |
| `client_hash_id`, `content_hash_id` | **Context** | Joins and the grouped train/test split only, never features |
| `gsc_data_available`, `ga4_data_available` | **Context** | Availability filters, not signals |
| `content_updated_date`, `last_optimized_date` | **Excluded** | Snapshot values from July 2026 that can leak edits made after the decision moment |
| GA4 columns (`ga4_sessions`, …) | **Excluded** | Only present for clients after their GA4 start; this slice is GSC-only so every page is measured the same way |
| `keyword_hash_id`, `url_hash_id` | **Excluded** | Pseudonyms with no meaning; using them invites memorising instead of learning |

The cell below shows that the fields I use exist in the March partition, with their types.

In [3]:
con.sql(f"DESCRIBE SELECT * FROM {MONTH}").df()[['column_name', 'column_type']]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1: the grain

Claim: in the raw table one row is one page on one day, so `report_date + client_hash_id + content_hash_id` never repeats. If the grain holds, this query returns **zero rows**.

In [4]:
dupes = con.sql('''
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM march
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
''').df()
print(f'keys that appear more than once: {len(dupes)}')
dupes

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

keys that appear more than once: 0


,report_date,client_hash_id,content_hash_id,n


### Query 2: row count and date span of my slice

Claim: the slice covers every day of March 2026, 1 to 31.

In [5]:
con.sql('''
    SELECT COUNT(*)                        AS rows,
           MIN(report_date)                AS first_day,
           MAX(report_date)                AS last_day,
           COUNT(DISTINCT report_date)     AS days,
           COUNT(DISTINCT client_hash_id)  AS clients,
           COUNT(DISTINCT content_hash_id) AS pages
    FROM march
''').df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,first_day,last_day,days,clients,pages
0,9841378,2026-03-01,2026-03-31,31,55,331437


### Query 3: availability, filtered with `IS TRUE`

Claim: not every row has search data. A missing flag must not count as "available", so the filter is `IS TRUE`, which drops both FALSE and NULL.

In [6]:
avail = con.sql('''
    SELECT COUNT(*)                                                    AS all_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)          AS gsc_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)          AS ga4_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                              AND ga4_data_available IS TRUE)          AS both_rows
    FROM march
''').df()
avail['gsc_share'] = avail['gsc_rows'] / avail['all_rows']
avail['ga4_share'] = avail['ga4_rows'] / avail['all_rows']
avail

,all_rows,gsc_rows,ga4_rows,both_rows,gsc_share,ga4_share
0,9841378,3611061,413966,364347,0.366926,0.042064


### Five features, each with "available when?"

The frame is built only from rows with `gsc_data_available IS TRUE`. To keep noise out, a page needs at least 100 impressions in 1–15 March and must have been created before March.

| Feature | Knowable at the decision moment (16 March) because… |
|---|---|
| `imp_per_day` – average daily impressions, 1–15 March | …it only adds up days before 16 March |
| `ctr` – clicks ÷ impressions, 1–15 March | …both counts come from days before 16 March |
| `avg_position` – impression-weighted Google position, 1–15 March | …it is averaged over days before 16 March |
| `momentum` – log of (daily impressions 8–15 March ÷ daily impressions 1–7 March) | …both halves end before 16 March. It shows whether the page was already sliding |
| `age_days` – days from creation to 15 March | …the creation date is fixed long before the decision |

In [7]:
frame = con.sql('''
    WITH d AS (
        SELECT * FROM march WHERE gsc_data_available IS TRUE
    ),
    pages AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15')                          AS imp_h1,
               SUM(gsc_clicks)      FILTER (WHERE report_date <= DATE '2026-03-15')                          AS clk_h1,
               SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-07')                          AS imp_w1,
               SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '2026-03-08' AND DATE '2026-03-15') AS imp_w2,
               SUM(gsc_avg_position * gsc_impressions)
                   FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0)                    AS pos_x_imp_h1,
               SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-16')                          AS imp_h2
        FROM d
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15') >= 100
    )
    SELECT p.*, c.content_created_date
    FROM pages p
    JOIN content c USING (client_hash_id, content_hash_id)
    WHERE c.content_created_date < DATE '2026-03-01'
''').df()

frame['imp_h2'] = frame['imp_h2'].fillna(0)
frame['imp_per_day']  = frame['imp_h1'] / 15
frame['ctr']          = frame['clk_h1'] / frame['imp_h1']
frame['avg_position'] = frame['pos_x_imp_h1'] / frame['imp_h1']
frame['momentum']     = np.log((frame['imp_w2'].fillna(0) / 8 + 1) / (frame['imp_w1'].fillna(0) / 7 + 1))
frame['age_days']     = (pd.Timestamp('2026-03-15') - pd.to_datetime(frame['content_created_date'])).dt.days

# Label: the future half of March (16-31 March, 16 days).
frame['is_declining'] = ((frame['imp_h2'] / 16) < 0.8 * frame['imp_per_day']).astype(int)

FEATURES = ['imp_per_day', 'ctr', 'avg_position', 'momentum', 'age_days']

assert not frame.duplicated(['client_hash_id', 'content_hash_id']).any(), 'frame grain broken'
print(f'pages in the frame: {len(frame):,} from {frame.client_hash_id.nunique()} clients')
print(f'declining share (label = 1): {frame.is_declining.mean():.1%}')
frame[FEATURES + ['is_declining']].describe().round(3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages in the frame: 74,377 from 36 clients
declining share (label = 1): 33.6%


,imp_per_day,ctr,avg_position,momentum,age_days,is_declining
count,74377.000,74377.000,74377.000,74377.000,74377.000,74377.000
mean,110.989,0.003,12.505,0.005,189.432,0.336
std,242.673,0.004,12.896,0.685,120.003,0.472
min,6.667,0.000,0.014,-7.806,15.000,0.000
25%,16.267,0.000,4.290,-0.299,69.000,0.000
50%,39.933,0.001,7.355,-0.047,180.000,0.000
75%,106.467,0.004,16.093,0.213,250.000,1.000
max,10771.667,0.214,119.430,6.537,478.000,1.000


### A quick, honest score

Pages from the same client move together, so the split is **by client**: the model is tested on clients it never saw. Two numbers: ROC-AUC (0.5 = guessing) and Precision@50 (share of the top-50 ranked pages that really declined), next to the base rate.

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
                           .split(frame, groups=frame['client_hash_id']))
train, test = frame.iloc[train_idx], frame.iloc[test_idx]

def quick_score(cols):
    model = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, random_state=42, n_jobs=-1)
    model.fit(train[cols], train['is_declining'])
    p = model.predict_proba(test[cols])[:, 1]
    top50 = test.assign(p=p).nlargest(50, 'p')
    return {'roc_auc': round(roc_auc_score(test['is_declining'], p), 3),
            'precision_at_50': round(top50['is_declining'].mean(), 3)}

print(f'train: {len(train):,} pages / {train.client_hash_id.nunique()} clients   '
      f'test: {len(test):,} pages / {test.client_hash_id.nunique()} clients')
print(f'base rate in test: {test.is_declining.mean():.3f}')
honest = quick_score(FEATURES)
print('honest score (5 features):', honest)

train: 48,407 pages / 27 clients   test: 25,970 pages / 9 clients
base rate in test: 0.333
honest score (5 features): {'roc_auc': np.float64(0.721), 'precision_at_50': np.float64(0.9)}


### The trap: one label-derived column, on purpose

`imp_h2_per_day` is the page's average daily impressions in **16–31 March**, the same window the label is computed from. Nobody can know it on 16 March. Adding it should push the score toward perfect. **This experiment is invalid by design and is not part of the model.**

In [9]:
frame['imp_h2_per_day'] = frame['imp_h2'] / 16
train, test = frame.iloc[train_idx], frame.iloc[test_idx]

leaky = quick_score(FEATURES + ['imp_h2_per_day'])
print('LEAKY score (5 features + imp_h2_per_day):', leaky, '  <- invalid, never use')

# Delete the leak and keep the honest number.
frame = frame.drop(columns=['imp_h2_per_day'])
train, test = frame.iloc[train_idx], frame.iloc[test_idx]
honest_again = quick_score(FEATURES)
print('honest score after deleting it:', honest_again)
assert 'imp_h2_per_day' not in frame.columns

LEAKY score (5 features + imp_h2_per_day): {'roc_auc': np.float64(0.99), 'precision_at_50': np.float64(1.0)}   <- invalid, never use
honest score after deleting it: {'roc_auc': np.float64(0.721), 'precision_at_50': np.float64(0.9)}


**What happened:** RESULT_PLACEHOLDER

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

LIMIT_PLACEHOLDER

In [10]:
share = frame['client_hash_id'].value_counts(normalize=True)
print(f'clients in the frame: {len(share)}')
print(f'share of pages from the biggest client: {share.iloc[0]:.1%}')
print(f'share of pages from the top 3 clients: {share.iloc[:3].sum():.1%}')

clients in the frame: 36
share of pages from the biggest client: 23.6%
share of pages from the top 3 clients: 56.0%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.